# **Homework 2: Phoneme Classification**


Objectives:
* Solve a classification problem with deep neural networks (DNNs).
* Understand recursive neural networks (RNNs).

If you have any questions, please contact the TAs via TA hours, NTU COOL, or email to mlta-2023-spring@googlegroups.com

# Some Utility Functions
**Fixes random number generator seeds for reproducibility.**

In [2]:
import numpy as np
import torch
import random

def same_seeds(seed):
    random.seed(seed) 
    np.random.seed(seed)  
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed) 
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

**Helper functions to pre-process the training data from raw MFCC features of each utterance.**

A phoneme may span several frames and is dependent to past and future frames. \
Hence we concatenate neighboring phonemes for training to achieve higher accuracy. The **concat_feat** function concatenates past and future k frames (total 2k+1 = n frames), and we predict the center frame.

Feel free to modify the data preprocess functions, but **do not drop any frame** (if you modify the functions, remember to check that the number of frames are the same as mentioned in the slides)

In [3]:
import os
import torch
from tqdm import tqdm

def load_feat(path):
    feat = torch.load(path)
    return feat

def shift(x, n):
    if n < 0:
        left = x[0].repeat(-n, 1)
        right = x[:n]
    elif n > 0:
        right = x[-1].repeat(n, 1)
        left = x[n:]
    else:
        return x

    return torch.cat((left, right), dim=0)

# def concat_feat(x, concat_n):
#     assert concat_n % 2 == 1 # n must be odd
#     if concat_n < 2:
#         return x
#     seq_len, feature_dim = x.size(0), x.size(1)
#     x = x.repeat(1, concat_n) 
#     x = x.view(seq_len, concat_n, feature_dim).permute(1, 0, 2) # concat_n, seq_len, feature_dim
#     mid = (concat_n // 2)
#     for r_idx in range(1, mid+1):
#         x[mid + r_idx, :] = shift(x[mid + r_idx], r_idx)
#         x[mid - r_idx, :] = shift(x[mid - r_idx], -r_idx)

#     return x.permute(1, 0, 2).view(seq_len, concat_n * feature_dim)

# def preprocess_data(split, feat_dir, phone_path, concat_nframes, train_ratio=0.8):
#     class_num = 41 # NOTE: pre-computed, should not need change

#     if split == 'train' or split == 'val':
#         mode = 'train'
#     elif split == 'test':
#         mode = 'test'
#     else:
#         raise ValueError('Invalid \'split\' argument for dataset: PhoneDataset!')

#     label_dict = {}
#     if mode == 'train':
#         for line in open(os.path.join(phone_path, f'{mode}_labels.txt')).readlines():
#             line = line.strip('\n').split(' ')
#             label_dict[line[0]] = [int(p) for p in line[1:]]
        
#         # split training and validation data
#         usage_list = open(os.path.join(phone_path, 'train_split.txt')).readlines()
#         random.shuffle(usage_list)
#         train_len = int(len(usage_list) * train_ratio)
#         usage_list = usage_list[:train_len] if split == 'train' else usage_list[train_len:]

#     elif mode == 'test':
#         usage_list = open(os.path.join(phone_path, 'test_split.txt')).readlines()

#     usage_list = [line.strip('\n') for line in usage_list]
#     print('[Dataset] - # phone classes: ' + str(class_num) + ', number of utterances for ' + split + ': ' + str(len(usage_list)))

#     max_len = 3000000
#     X = torch.empty(max_len, 39 * concat_nframes)
#     if mode == 'train':
#         y = torch.empty(max_len, dtype=torch.long)

#     idx = 0
#     for i, fname in tqdm(enumerate(usage_list)):
#         feat = load_feat(os.path.join(feat_dir, mode, f'{fname}.pt'))
#         cur_len = len(feat)
#         feat = concat_feat(feat, concat_nframes)
#         if mode == 'train':
#           label = torch.LongTensor(label_dict[fname])

#         X[idx: idx + cur_len, :] = feat
#         if mode == 'train':
#           y[idx: idx + cur_len] = label

#         idx += cur_len

#     X = X[:idx, :]
#     if mode == 'train':
#       y = y[:idx]

#     print(f'[INFO] {split} set')
#     print(X.shape)
#     if mode == 'train':
#       print(y.shape)
#       return X, y
#     else:
#       return X
def preprocess_data(split, feat_dir, phone_path, train_ratio=0.75):
    label_dict = {}
    if split in ('train', 'val'):
        mode = 'train'
        for line in open(os.path.join(phone_path, 'train_labels.txt')):
            p = line.strip().split(' ')
            label_dict[p[0]] = [int(x) for x in p[1:]]
        usage = [l.strip() for l in open(os.path.join(phone_path, 'train_split.txt'))]
        random.shuffle(usage)
        k = int(len(usage) * train_ratio)
        usage = usage[:k] if split == 'train' else usage[k:]
    else:
        mode = 'test'
        usage = [l.strip() for l in open(os.path.join(phone_path, 'test_split.txt'))]

    X, Y = [], []
    for name in tqdm(usage):
        X.append(torch.load(os.path.join(feat_dir, mode, f'{name}.pt')))   # (T, 39)
        if mode == 'train':
            Y.append(torch.LongTensor(label_dict[name]))                    # (T,)
    return (X, Y) if mode == 'train' else X


# Dataset

In [4]:
# import torch
# from torch.utils.data import Dataset

# class LibriDataset(Dataset):
#     def __init__(self, X, y=None):
#         self.data = X
#         if y is not None:
#             self.label = torch.LongTensor(y)
#         else:
#             self.label = None

#     def __getitem__(self, idx):
#         if self.label is not None:
#             return self.data[idx], self.label[idx]
#         else:
#             return self.data[idx]

#     def __len__(self):
#         return len(self.data)
from torch.utils.data import Dataset
import torch.nn as nn

class LibriDataset(Dataset):
    def __init__(self, X, y=None):
        self.X, self.y = X, y
    def __len__(self):
        return len(self.X)
    def __getitem__(self, i):
        return (self.X[i], self.y[i]) if self.y is not None else self.X[i]

def collate_fn(batch):
    if isinstance(batch[0], tuple):
        xs, ys = zip(*batch)
        lengths = torch.LongTensor([len(x) for x in xs])
        X = nn.utils.rnn.pad_sequence(xs, batch_first=True)
        Y = nn.utils.rnn.pad_sequence(ys, batch_first=True)
        return X, Y, lengths
    lengths = torch.LongTensor([len(x) for x in batch])
    X = nn.utils.rnn.pad_sequence(batch, batch_first=True)
    return X, lengths


# Model
Feel free to modify the structure of the model.

In [5]:
# import torch.nn as nn

# class BasicBlock(nn.Module):
#     def __init__(self, input_dim, output_dim):
#         super(BasicBlock, self).__init__()

#         self.block = nn.Sequential(
#             nn.Linear(input_dim, output_dim),
#             nn.BatchNorm1d(output_dim),
#             nn.ReLU(),
#             nn.Dropout(0.3),
#         )

#     def forward(self, x):
#         x = self.block(x)
#         return x


# class Classifier(nn.Module):
#     def __init__(self, input_dim, output_dim=41, hidden_layers=1, hidden_dim=256):
#         super(Classifier, self).__init__()

#         self.fc = nn.Sequential(
#             BasicBlock(input_dim, hidden_dim),
#             *[BasicBlock(hidden_dim, hidden_dim) for _ in range(hidden_layers)],
#             nn.Linear(hidden_dim, output_dim)
#         )

#     def forward(self, x):
#         x = self.fc(x)
#         return x
class Classifier(nn.Module):
    def __init__(self, input_dim=39, hidden_dim=256, num_layers=2, output_dim=41):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers=num_layers,
                            batch_first=True, bidirectional=True,
                            dropout=0.3 if num_layers > 1 else 0)
        self.fc = nn.Linear(hidden_dim * 2, output_dim)

    def forward(self, x, lengths):
        packed = nn.utils.rnn.pack_padded_sequence(
            x, lengths.cpu(), batch_first=True, enforce_sorted=False)
        out, _ = self.lstm(packed)
        out, _ = nn.utils.rnn.pad_packed_sequence(out, batch_first=True)
        return self.fc(out)      # (B, T, 41)


# Hyper-parameters

In [10]:
# # data prarameters
# concat_nframes = 17              # the number of frames to concat with, n must be odd (total 2k+1 = n frames)
# train_ratio = 0.75               # the ratio of data used for training, the rest will be used for validation

# # training parameters
# seed = 1213                        # random seed
# batch_size = 512                # batch size
# num_epoch = 60                   # the number of training epoch
# learning_rate = 1e-4         # learning rate
# model_path = './model.ckpt'     # the path where the checkpoint will be saved

# # model parameters
# input_dim = 39 * concat_nframes # the input dim of the model, you should not change the value
# hidden_layers = 8               # the number of hidden layers
# hidden_dim = 1024                # the hidden dim
train_ratio = 0.75
seed = 1213
batch_size = 64
num_epoch = 40
learning_rate = 1e-3
model_path = './model_rnn.ckpt'
hidden_dim = 512
num_layers = 3


# Dataloader

In [11]:
# from torch.utils.data import DataLoader
# import gc
from torch.utils.data import DataLoader

# same_seeds(seed)
# device = 'cuda' if torch.cuda.is_available() else 'cpu'
# print(f'DEVICE: {device}')

# # preprocess data
# train_X, train_y = preprocess_data(split='train', feat_dir='C:/Users/35433/Desktop/Claude ML/ML-Setup/Hw_2/libriphone/feat', phone_path='C:/Users/35433/Desktop/Claude ML/ML-Setup/Hw_2/libriphone', concat_nframes=concat_nframes, train_ratio=train_ratio)
# same_seeds(seed) 
# val_X, val_y = preprocess_data(split='val', feat_dir='C:/Users/35433/Desktop/Claude ML/ML-Setup/Hw_2/libriphone/feat', phone_path='C:/Users/35433/Desktop/Claude ML/ML-Setup/Hw_2/libriphone', concat_nframes=concat_nframes, train_ratio=train_ratio)

# # get dataset
# train_set = LibriDataset(train_X, train_y)
# val_set = LibriDataset(val_X, val_y)

# # remove raw feature to save memory
# del train_X, train_y, val_X, val_y
# gc.collect()

# # get dataloader
# train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True)
# val_loader = DataLoader(val_set, batch_size=batch_size, shuffle=False)
import gc

feat_dir = 'C:/Users/35433/Desktop/Claude ML/ML-Setup/Hw_2/libriphone/feat'
phone_path = 'C:/Users/35433/Desktop/Claude ML/ML-Setup/Hw_2/libriphone'

same_seeds(seed)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'DEVICE: {device}')

train_X, train_y = preprocess_data('train', feat_dir, phone_path, train_ratio)
same_seeds(seed)          # 重新设种子，保证 train/val 用同一个打乱结果，不重叠
val_X, val_y = preprocess_data('val', feat_dir, phone_path, train_ratio)

train_set = LibriDataset(train_X, train_y)
val_set = LibriDataset(val_X, val_y)

train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_set, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)


DEVICE: cuda


100%|██████████| 858/858 [00:00<00:00, 1704.73it/s]


# Training

In [12]:
# # create model, define a loss function, and optimizer
# model = Classifier(input_dim=input_dim, hidden_layers=hidden_layers, hidden_dim=hidden_dim).to(device)
# criterion = nn.CrossEntropyLoss() 
# optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

# best_acc = 0.0
# for epoch in range(num_epoch):
#     train_acc = 0.0
#     train_loss = 0.0
#     val_acc = 0.0
#     val_loss = 0.0
    
#     # training
#     model.train() # set the model to training mode
#     for i, batch in enumerate(tqdm(train_loader)):
#         features, labels = batch
#         features = features.to(device)
#         labels = labels.to(device)
        
#         optimizer.zero_grad() 
#         outputs = model(features) 
        
#         loss = criterion(outputs, labels)
#         loss.backward() 
#         optimizer.step() 
        
#         _, train_pred = torch.max(outputs, 1) # get the index of the class with the highest probability
#         train_acc += (train_pred.detach() == labels.detach()).sum().item()
#         train_loss += loss.item()
    
#     # validation
#     model.eval() # set the model to evaluation mode
#     with torch.no_grad():
#         for i, batch in enumerate(tqdm(val_loader)):
#             features, labels = batch
#             features = features.to(device)
#             labels = labels.to(device)
#             outputs = model(features)
            
#             loss = criterion(outputs, labels) 
            
#             _, val_pred = torch.max(outputs, 1) 
#             val_acc += (val_pred.cpu() == labels.cpu()).sum().item() # get the index of the class with the highest probability
#             val_loss += loss.item()

#     print(f'[{epoch+1:03d}/{num_epoch:03d}] Train Acc: {train_acc/len(train_set):3.5f} Loss: {train_loss/len(train_loader):3.5f} | Val Acc: {val_acc/len(val_set):3.5f} loss: {val_loss/len(val_loader):3.5f}')

#     # if the model improves, save a checkpoint at this epoch
#     if val_acc > best_acc:
#         best_acc = val_acc
#         torch.save(model.state_dict(), model_path)
#         print(f'saving model with acc {best_acc/len(val_set):.5f}')
model = Classifier(hidden_dim=hidden_dim, num_layers=num_layers).to(device)
criterion = nn.CrossEntropyLoss(reduction='none')
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

best_acc = 0.0

for epoch in range(num_epoch):
    # ---------- 训练 ----------
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for X, Y, lengths in train_loader:
        X, Y, lengths = X.to(device), Y.to(device), lengths.to(device)
        mask = torch.arange(Y.size(1), device=device)[None, :] < lengths[:, None]

        out = model(X, lengths)                       # (B, T, 41)
        loss = criterion(out.reshape(-1, 41), Y.reshape(-1))
        loss = (loss * mask.reshape(-1)).sum() / mask.sum()

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        pred = out.argmax(-1)
        correct += ((pred == Y) & mask).sum().item()
        total += mask.sum().item()
        total_loss += loss.item()

    print(f"[{epoch+1:02d}/{num_epoch}] train loss {total_loss/len(train_loader):.4f} acc {correct/total:.4f}", end="")

    # ---------- 验证 ----------
    model.eval()
    v_correct, v_total = 0, 0
    with torch.no_grad():
        for X, Y, lengths in val_loader:
            X, Y, lengths = X.to(device), Y.to(device), lengths.to(device)
            mask = torch.arange(Y.size(1), device=device)[None, :] < lengths[:, None]
            out = model(X, lengths)
            v_correct += ((out.argmax(-1) == Y) & mask).sum().item()
            v_total += mask.sum().item()

    val_acc = v_correct / v_total
    print(f" | val acc {val_acc:.5f}")

    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), model_path)
        print(f"    -> saving model with val acc {best_acc:.5f}")


[01/40] train loss 2.1347 acc 0.4063 | val acc 0.54823
    -> saving model with val acc 0.54823
[02/40] train loss 1.2597 acc 0.6144 | val acc 0.66988
    -> saving model with val acc 0.66988
[03/40] train loss 0.9836 acc 0.6949 | val acc 0.70979
    -> saving model with val acc 0.70979
[04/40] train loss 0.8301 acc 0.7412 | val acc 0.74422
    -> saving model with val acc 0.74422
[05/40] train loss 0.7225 acc 0.7736 | val acc 0.75826
    -> saving model with val acc 0.75826


KeyboardInterrupt: 

In [ ]:
del train_set, val_set
del train_loader, val_loader
gc.collect()

9

# Testing
Create a testing dataset, and load model from the saved checkpoint.

In [9]:
# # load data
# test_X = preprocess_data(split='test', feat_dir='C:/Users/35433/Desktop/Claude ML/ML-Setup/Hw_2/libriphone/feat', phone_path='C:/Users/35433/Desktop/Claude ML/ML-Setup/Hw_2/libriphone', concat_nframes=concat_nframes)
# test_set = LibriDataset(test_X, None)
# test_loader = DataLoader(test_set, batch_size=batch_size, shuffle=False)
# # load model
# model = Classifier(input_dim=input_dim, hidden_layers=hidden_layers, hidden_dim=hidden_dim).to(device)
# model.load_state_dict(torch.load(model_path))
# # Make prediction.
# pred = np.array([], dtype=np.int32)

# model.eval()
# with torch.no_grad():
#     for i, batch in enumerate(tqdm(test_loader)):
#         features = batch
#         features = features.to(device)

#         outputs = model(features)

#         _, test_pred = torch.max(outputs, 1) # get the index of the class with the highest probability
#         pred = np.concatenate((pred, test_pred.cpu().numpy()), axis=0)

# # Write prediction to a CSV file.

# # After finish running this block, download the file `prediction.csv` from the files section on the left-hand side and submit it to Kaggle.
# with open('prediction.csv', 'w') as f:
#     f.write('Id,Class\n')
#     for i, y in enumerate(pred):
#         f.write('{},{}\n'.format(i, y))
test_X = preprocess_data('test', feat_dir, phone_path)
test_set = LibriDataset(test_X)
test_loader = DataLoader(test_set, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)

model = Classifier(hidden_dim=hidden_dim, num_layers=num_layers).to(device)
model.load_state_dict(torch.load(model_path))
model.eval()

pred = []
with torch.no_grad():
    for X, lengths in test_loader:
        X, lengths = X.to(device), lengths.to(device)
        out = model(X, lengths).argmax(-1)
        for i, L in enumerate(lengths):
            pred.extend(out[i, :L].cpu().tolist())
# 写提交文件
with open('prediction.csv', 'w', newline='') as f:
    f.write('Id,Class\n')
    for i, y in enumerate(pred):
        f.write('{},{}\n'.format(i, y))

print(f'总共 {len(pred)} 帧，已写入 prediction.csv')


100%|██████████| 857/857 [00:00<00:00, 1584.65it/s]


总共 527364 帧，已写入 prediction.csv


# 我的实验记录

每次改了超参数就在下面手动追加一行，记录改了什么、Val Acc 多少、我的猜测对不对。

In [ ]:
# 格式：改了什么 -> Val Acc -> 猜测是否成立
# 示例：concat_nframes 3->3 (基线) -> Val Acc 0.50041

log = []

log.append({
    "改动": "基线 concat_nframes=3, hidden_layers=2, hidden_dim=64, epoch=10",
    "val_acc": 0.50041,
    "猜测": "",
    "结果是否符合猜测": "",

    "第一次改动": "concat_nframes3->5",
    "val_acc": 0.53863,
    "猜测": "better",
    "结果是否符合猜测": "yes",

    "第二次改动": "hidden_layer2->4",
    "val_acc": 0.53188,
    "猜测": "better",
    "结果是否符合猜测": "并没有",

    "第三次改动": "hidden_layer2->4, hidden_dim=128",
    "val_acc": 0.56636,
    "猜测": "better",
    "结果是否符合猜测": "yes",

    "第四次改动": "hidden_layer2->4, hidden_dim=128, num_epoch10->40, learning rate 1e-4->1e-5",
    "val_acc": 0.53447,
    "猜测": "better",
    "结果是否符合猜测": "not yet  怀疑是因为学习率太低轮次太少不能到达最低点，同时时间花费太多",

    "第五次改动": "hidden_dim=256->512, concat_nframes = 9->15",
    "val_acc": 0.80434,
    "猜测": "better",
    "结果是否符合猜测": "好的批爆！",

    "第六次改动": "hidden_layer4->6",
    "val_acc": 0.81143,
    "猜测": "better",
    "结果是否符合猜测": "变好效果一般",

    "第七次改动": "learning_rate 1e-4->1e-5, epoch->60",
    "val_acc": 0.69414,
    "猜测": "better",
    "结果是否符合猜测": "反而变得更差，调高epoch",

    "第八次改动": "epoch->100",
    "val_acc": 0.72024,
    "猜测": "better",
    "结果是否符合猜测": "也很一般，时间太久半个小时",

    "第九次改动": "epoch->60, learning_rate=1e-4, hidden_layer->8, hidden_dim->1024",
    "val_acc": 0.90734,
    "猜测": "better",
    "结果是否符合猜测": "非常不错！",

    #模型出现了过拟合现象，而且争取率和kaggle不一样，先进行改动使得正确率是真的

    "第十次改动": "model一格加上BatchNormld",
    "val_acc": 0.68337,
    "猜测": "better",
    "结果是否符合猜测": "也很一般，时间太久半个小时，BN收益不明显",

    "第十一次改动": "model一格再加上dropout",
    "val_acc": 0.74325,
    "猜测": "better",
    "结果是否符合猜测": "趋势明显，时间半个小时，增加近六个点",

    "第十二次改动": "从MLP改成RNN_____新建副本——hw2-RNN",
    "val_acc": 0.764,
    "猜测": "better",
    "结果是否符合猜测": "趋势一般，五分钟，加大epoch20->40",

    "第十三次改动": "epoch20->40",
    "val_acc": 0.76110,
    "猜测": "better",
    "结果是否符合猜测": "呵呵，20轮其实已经收敛了，再调高也没意义",

    "第十四次改动": "epoch20->40, hidden_dim = 256 → 512",
    "val_acc": 0.78356,
    "猜测": "better",
    "结果是否符合猜测": "还算不错，二十分钟",

    "第十五次改动": "num_layers2->3",
    "val_acc":  0,
    "猜测": "better",
    "结果是否符合猜测": "时间太长了，扛不住",
    
})


for r in log:
    print(r)


{'改动': '基线 concat_nframes=3, hidden_layers=2, hidden_dim=64, epoch=10', 'val_acc': 0.72024, '猜测': 'better', '结果是否符合猜测': '也很一般', '第一次改动': 'concat_nframes3->5', '第二次改动': 'hidden_layer2->4', '第三次改动': 'hidden_layer2->4, hidden_dim=128', '第四次改动': 'hidden_layer2->4, hidden_dim=128, num_epoch10->40, learning rate 1e-4->1e-5', '第五次改动': 'hidden_dim=256->512, concat_nframes = 9->15', '第六次改动': 'hidden_layer4->6', '第七次改动': 'epoch->100'}
